In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# Locate resumes_clean.csv 
path = Path("data/processed/resumes_clean.csv")
if not path.exists():
    path = Path("../data/processed/resumes_clean.csv")

df = pd.read_csv(path)
print(f"Loaded {len(df)} resumes across {df['Category'].nunique()} categories.")

X = df['Resume_Clean'].fillna('')
y = df['Category']

# 80/20 stratified split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training samples:", len(X_train), "| Testing samples:", len(X_test))

Loaded 962 resumes across 25 categories.
Training samples: 769 | Testing samples: 193


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# 1. TF-IDF and Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=42))
])

# 2. Train model
print("Training classifier...")
pipeline.fit(X_train, y_train)

# 3. Test model
y_pred = pipeline.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred))

Training classifier...

Accuracy: 0.9948

                           precision    recall  f1-score   support

                 Advocate       1.00      1.00      1.00         4
                     Arts       1.00      1.00      1.00         7
       Automation Testing       0.83      1.00      0.91         5
               Blockchain       1.00      1.00      1.00         8
         Business Analyst       1.00      1.00      1.00         6
           Civil Engineer       1.00      1.00      1.00         5
             Data Science       1.00      1.00      1.00         8
                 Database       1.00      1.00      1.00         7
          DevOps Engineer       1.00      0.91      0.95        11
         DotNet Developer       1.00      1.00      1.00         5
            ETL Developer       1.00      1.00      1.00         8
   Electrical Engineering       1.00      1.00      1.00         6
                       HR       1.00      1.00      1.00         9
                   

In [ ]:
from pathlib import Path
import joblib

models_dir = Path("models")
if not models_dir.exists():
    models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Export the trained classifier and vectorizer
joblib.dump(pipeline.named_steps['clf'], models_dir / "classifier.pkl")
joblib.dump(pipeline.named_steps['tfidf'], models_dir / "tfidf_vectorizer.pkl")

print(f"Artifacts successfully saved to {models_dir.resolve()}/")

Artifacts successfully saved to /workspaces/SmartHire/models/


In [4]:
sample_resume = """
Experienced in Python, pandas, numpy, scikit-learn, TensorFlow,
data cleaning, exploratory data analysis, machine learning algorithms,
and predictive modeling. Strong background in statistics and SQL.
"""

# Transform and predict
vec_sample = pipeline.named_steps['tfidf'].transform([sample_resume.lower()])
predicted_domain = pipeline.named_steps['clf'].predict(vec_sample)[0]
confidence = pipeline.named_steps['clf'].predict_proba(vec_sample).max()

print(f"Predicted Domain: {predicted_domain}")
print(f"Confidence Score: {confidence * 100:.2f}%")

Predicted Domain: Data Science
Confidence Score: 32.33%


In [5]:
import numpy as np

probs = pipeline.named_steps['clf'].predict_proba(vec_sample)[0]
top_3_idx = np.argsort(probs)[-3:][::-1]

for idx in top_3_idx:
    print(f"{pipeline.classes_[idx]}: {probs[idx] * 100:.2f}%")

Data Science: 32.33%
Python Developer: 6.04%
Hadoop: 4.16%
